# Spark Preparation
We check if we are in Google Colab.  If this is the case, install all necessary packages.

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 3.3.2 with hadoop 3.3, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab.
Learn more from [A Must-Read Guide on How to Work with PySpark on Google Colab for Data Scientists!](https://www.analyticsvidhya.com/blog/2020/11/a-must-read-guide-on-how-to-work-with-pyspark-on-google-colab-for-data-scientists/)

In [97]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [98]:
if IN_COLAB:
    !apt-get install openjdk-8-jdk-headless -qq > /dev/null
    !wget -q https://dlcdn.apache.org/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
    !tar xf spark-3.3.2-bin-hadoop3.tgz
    !mv spark-3.3.2-bin-hadoop3 spark
    !pip install -q findspark
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["SPARK_HOME"] = "/content/spark"

# Start a Local Cluster

In [99]:
from functools import reduce
from pyspark.sql.functions import (col, trim, lower, regexp_replace, sum, udf, to_timestamp,split, datediff, substring, length,
    current_timestamp, when, datediff, try_to_timestamp, to_date)
from pythainlp import word_tokenize
from pyspark.sql.types import ArrayType, StringType
from pythainlp.corpus import thai_stopwords



In [100]:
spark_url = 'local'
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder \
    .appName("TraffyFondueDataCleaning") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [101]:
sc = spark.sparkContext

file_path = r'C:\Users\sasit\CU\2-1\dsde\project\dsdengdeng-project-dsde\data\raw\traffy-fondue\bangkok_2025-12-new.csv'

## schema

In [102]:

traffy_schema = StructType([
    # ตัวระบุเฉพาะ
    StructField("ticket_id", StringType(), True),
    
    # ข้อมูลปัญหาและการจัดการ
    StructField("type", StringType(), True),         # หมวดหมู่ปัญหา
    StructField("organization", StringType(), True), # หน่วยงานที่รับผิดชอบ
    StructField("comment", StringType(), True),      # ข้อความร้องเรียน (สำคัญสำหรับ LLM)
    StructField("photo", StringType(), True),
    StructField("photo_after", StringType(), True),
    
    # ข้อมูลพิกัดและตำแหน่ง
    StructField("coords", StringType(), True),       # พิกัด Lat/Long (เก็บเป็น String ก่อนแล้วค่อย Parse)
    StructField("address", StringType(), True),
    StructField("subdistrict", StringType(), True),
    StructField("district", StringType(), True),
    StructField("province", StringType(), True),
    
    # ข้อมูลเวลาและสถานะ
    StructField("timestamp", StringType(), True),    # วันที่สร้าง (เก็บเป็น String ก่อนแล้วค่อย Cast เป็น Timestamp)
    StructField("state", StringType(), True),        # สถานะปัจจุบัน (ใช้ในการ Filter Active Issues)
    
    # ข้อมูลการตอบรับและกิจกรรม
    StructField("star", FloatType(), True),          # เรทติ้ง 0-5
    StructField("count_reopen", IntegerType(), True), # จำนวนครั้งที่เปิดซ้ำ
    StructField("last_activity", StringType(), True)  # วันที่กิจกรรมล่าสุด (เก็บเป็น String ก่อน)
])

In [103]:
df_traffy = spark.read.csv(
    file_path,
    header=True,
    schema=traffy_schema,
    multiLine=True, # สำคัญ: หาก 'comment' หรือ 'address' มีหลายบรรทัด
    escape='"' # สำคัญ: หากมีเครื่องหมายคำพูดในข้อความ
)

In [104]:
df_traffy.select("last_activity").show(10, False)


+--------------------------+
|last_activity             |
+--------------------------+
|2025-12-03 12:19:45.637589|
|2025-12-02 15:24:52.286411|
|2025-12-01 20:29:04.962598|
|2025-12-01 09:35:52.278532|
|2025-12-01 21:04:10.36225 |
|2025-12-03 12:48:18.689521|
|2025-12-01 14:43:36.69552 |
|2025-12-01 08:49:21.628257|
|2025-12-02 09:11:39.915313|
|2025-12-01 17:57:19.669545|
+--------------------------+
only showing top 10 rows


In [105]:
# print(f"จำนวนแถวเริ่มต้น: {df_traffy.count()}")
# df_traffy.printSchema()

In [106]:


# ลิสต์หมวดหมู่หลักที่ส่งผลต่อมูลค่าอสังหาฯ และความน่าอยู่
livability_types = [
    "ถนน",
    "ทางเท้า",
    "ความปลอดภัย",
    "แสงสว่าง",
    "ความสะอาด",
    "กีดขวาง",
    "ท่อระบายน้ำ",
    "น้ำท่วม",
    "ต้นไม้",
    "PM2",
    "จราจร",
    "สะพาน"
]

# กรองข้อมูลตาม 'type' (หมวดหมู่ปัญหา)
# ใช้วิธี 'isin' ที่ตรงไปตรงมาที่สุด
df_filtered_type = df_traffy.filter(
    reduce(lambda a, b: a | b, [col("type").contains(t) for t in livability_types])
)




active_states = [
    "กำลังดำเนินการ",
    "รอรับเรื่อง" 
]

from pyspark.sql.functions import trim, col

df_filtered_type = df_filtered_type.withColumn(
    "state", trim(col("state"))
)

df_filtered_type = df_filtered_type.filter(
    col("state").isin(active_states)
)
print(f"จำนวนแถวหลังการกรองที่ใช้ในการวิเคราะห์: {df_filtered_type.count()}")



จำนวนแถวหลังการกรองที่ใช้ในการวิเคราะห์: 1675


In [107]:
df_final_spatial = df_filtered_type.withColumn(
    "lon_raw", 
    trim(split(col("coords"), ",").getItem(0)).cast("float")
).withColumn(
    "lat_raw", 
    trim(split(col("coords"), ",").getItem(1)).cast("float")
).withColumn(
    # Set the corrected columns
    "lon", col("lon_raw")
).withColumn(
    "lat", col("lat_raw")
).filter(
    # Filter for valid Thai coordinates (Roughly: lat between 5-21, lon between 97-105)
    (col("lat") >= 5) & (col("lat") <= 21) & 
    (col("lon") >= 97) & (col("lon") <= 105)
)
BANGKOK_PROVINCE_NAMES = ["กรุงเทพมหานคร", "กรุงเทพ","จังหวัดกรุงเทพมหานคร"]

# ใช้ df_final_spatial เป็น DataFrame ที่มีคอลัมน์ 'province'
df_bangkok_only = df_final_spatial.filter(
    col("province").isin(BANGKOK_PROVINCE_NAMES)
)

# ตรวจสอบจำนวนแถวหลังการกรอง
print(f"จำนวน Ticket ทั้งหมดในกรุงเทพมหานคร (รวมชื่อย่อ): {df_bangkok_only.count()}")

# ตรวจสอบว่าคอลัมน์ province เหลือแต่ชื่อที่เราต้องการเท่านั้น
print("การกระจายตัวของค่า 'province' หลังการกรอง:")
df_bangkok_only.groupBy("province").count().show()



จำนวน Ticket ทั้งหมดในกรุงเทพมหานคร (รวมชื่อย่อ): 1675
การกระจายตัวของค่า 'province' หลังการกรอง:
+-------------+-----+
|     province|count|
+-------------+-----+
|กรุงเทพมหานคร| 1675|
+-------------+-----+



In [108]:
# from pyspark.sql.functions import floor, count, round
# df_grid_prep = df_final_spatial.withColumn(
#     "grid_lat",
#     # (floor(lat * 100) / 100) จะตัดทศนิยมให้เหลือ 2 ตำแหน่ง (0.01, 0.02, ...)
#     floor(col("lat") * 100) / 100
# ).withColumn(
#     "grid_lon",
#     floor(col("lon") * 100) / 100
# )

# # --- 2. จัดกลุ่มและนับจำนวน ---
# # นับจำนวน Ticket ในแต่ละกริด
# df_coords_grouped = df_grid_prep.groupBy(
#     "grid_lat", 
#     "grid_lon"
# ).agg(
#     count("ticket_id").alias("incident_count")
# ).orderBy(col("incident_count").desc())

# # แสดง 20 พื้นที่ที่มีปัญหาหนาแน่นที่สุด (Top 20 Grid)
# print("Top 20 Grid Bins (0.01 deg) by Incident Count:")
# df_coords_grouped.show(20)

# # --- 3. ตรวจสอบความสะอาดของข้อมูล (Validation) ---
# # ตรวจสอบช่วงค่า Min/Max ของกริดที่สร้างขึ้น
# df_coords_grouped.describe().show()

In [109]:
df_trim_string = df_bangkok_only.withColumn(
    "timestamp_str", 
    substring(col("timestamp"), 1, 19) # เริ่มจาก index 1, เอา 19 ตัวอักษร
).withColumn(
    "last_activity_str", 
    substring(col("last_activity"), 1, 19) # ทำเหมือนกันกับ last_activity
)

# Format ที่ใช้หลังตัด:
TIMESTAMP_FORMAT_SIMPLE = "yyyy-MM-dd HH:mm:ss"

# 2. แปลง String เป็น Timestamp (TimestampType)
df_time_prep = df_trim_string.withColumn(
    "timestamp_dt", 
    to_timestamp(col("timestamp_str"), TIMESTAMP_FORMAT_SIMPLE)
).withColumn(
    "last_activity_dt", 
    to_timestamp(col("last_activity_str"), TIMESTAMP_FORMAT_SIMPLE)
)

# กรองแถวที่แปลง timestamp ไม่ได้ (Timestamp/last_activity เป็น NULL หลังแปลง)
df_time_prep = df_time_prep.filter(
    col("timestamp_dt").isNotNull() 
)

# 3. แปลงเป็น Date และคำนวณ DaysToFix
df_time_prep = df_time_prep.withColumn("timestamp_date", to_date(col("timestamp_dt")))
df_time_prep = df_time_prep.withColumn("last_activity_date", to_date(col("last_activity_dt")))

df_final_ready = df_time_prep.withColumn(
    "DaysToFix",
    when(
        # ถ้า state = 'เสร็จสิ้น'
        col("state") == "เสร็จสิ้น",
        datediff(col("last_activity_date"), col("timestamp_date"))
    ).otherwise(
        # ถ้า state = 'กำลังดำเนินการ' หรือ 'รอรับเรื่อง'
        datediff(to_date(current_timestamp()), col("timestamp_date"))
    )
)

# ตรวจสอบผลลัพธ์
df_final_ready.select(
    "ticket_id", "state", "lat", "lon", 
    "timestamp_dt", "last_activity_dt", "DaysToFix","timestamp_date", "last_activity_date"
).show(5, truncate=False)

+-----------+--------------+--------+---------+-------------------+-------------------+---------+--------------+------------------+
|ticket_id  |state         |lat     |lon      |timestamp_dt       |last_activity_dt   |DaysToFix|timestamp_date|last_activity_date|
+-----------+--------------+--------+---------+-------------------+-------------------+---------+--------------+------------------+
|2025-2W8ELM|กำลังดำเนินการ|13.73767|100.48559|2025-12-01 07:05:22|2025-12-01 21:04:10|5        |2025-12-01    |2025-12-01        |
|2025-C89Z9A|กำลังดำเนินการ|13.77285|100.48217|2025-12-01 07:48:22|2025-12-01 08:49:21|5        |2025-12-01    |2025-12-01        |
|2025-BPL6AD|กำลังดำเนินการ|13.7381 |100.65583|2025-12-01 07:58:37|2025-12-01 17:57:19|5        |2025-12-01    |2025-12-01        |
|P3WJDU     |กำลังดำเนินการ|13.71703|100.43573|2025-12-01 09:21:27|2025-12-01 10:47:28|5        |2025-12-01    |2025-12-01        |
|2025-RCYKJF|กำลังดำเนินการ|13.73823|100.59887|2025-12-01 09:41:37|2025-12-0

In [110]:

df_clean = (
    df_final_ready
    .withColumn("comment_clean", trim(col("comment")))
    .withColumn("comment_clean", lower(col("comment_clean")))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[\n\r\t]", " "))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[^ก-๙a-z0-9/. ]", ""))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), " +", " "))
)

MIN_COMMENT_LENGTH = 10
df_clean = df_clean.filter(
    (length(col("comment_clean")) >= MIN_COMMENT_LENGTH)
)
print(df_clean.count())
df_clean.show(20, truncate=False)

1667
+-----------+---------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [111]:
from pyspark.sql.functions import col, desc

# สมมติว่า df_final_ml คือ DataFrame ที่คุณใช้ล่าสุด

# 1. จัดกลุ่มตามคอลัมน์ comment_clean และนับจำนวนแถวในแต่ละกลุ่ม
df_comment_counts = df_clean.groupBy("comment_clean").count()

# 2. เรียงลำดับจากจำนวนนับ (count) ที่มากที่สุดไปน้อยที่สุด (Descending)
df_duplicate_summary = df_comment_counts.orderBy(
    col("count").desc()
)

# 3. แสดงผลลัพธ์ 20 อันดับแรก ที่มีจำนวนซ้ำกันมากกว่า 1 ครั้ง
print("--- 20 อันดับแรกของข้อความร้องเรียนที่ซ้ำกันมากที่สุด ---")
df_duplicate_summary.filter(col("count") > 1).show(
    20, 
    truncate=False # แสดงข้อความ comment_clean แบบเต็ม
)

--- 20 อันดับแรกของข้อความร้องเรียนที่ซ้ำกันมากที่สุด ---
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|comment_clean                                                                                                                                                                                                                                                                               |count|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|แจ้ง จักรยานยนต์ขับขี่บนทางเท้า ระหว่างซอยเพชรเกษม 62/1 62/2 6

In [112]:
from pyspark.sql.functions import col

# 1. กรองเฉพาะ Ticket ที่มี comment_clean ตรงกับ "ทางเท้าชำรุด"
comment_to_filter = "ทางเท้าชำรุด"
df_damaged_pavement = df_clean.filter(col("comment_clean") == comment_to_filter)

# 2. เลือกคอลัมน์ที่จำเป็นสำหรับการวิเคราะห์ความซ้ำซ้อนและการตอบคำถาม
df_analysis_subset = df_damaged_pavement.select(
    "ticket_id",
    "timestamp_dt", # เวลาที่แจ้งเรื่อง (ใช้ในการดูว่าแจ้งซ้ำในเวลาใกล้กันหรือไม่)
    "last_activity_dt", # เวลาที่มีความเคลื่อนไหวล่าสุด
    "lat",
    "lon",
    "district",
    "DaysToFix",
)

# 3. แสดงผลลัพธ์ 20 อันดับแรก (เรียงตามเวลาที่แจ้งล่าสุด)
print(f"--- 20 อันดับแรกของ Ticket ที่มี Comment: '{comment_to_filter}' ---")
df_analysis_subset.orderBy(col("timestamp_dt").desc()).show(20, truncate=False)

--- 20 อันดับแรกของ Ticket ที่มี Comment: 'ทางเท้าชำรุด' ---
+---------+------------+----------------+---+---+--------+---------+
|ticket_id|timestamp_dt|last_activity_dt|lat|lon|district|DaysToFix|
+---------+------------+----------------+---+---+--------+---------+
+---------+------------+----------------+---+---+--------+---------+



In [113]:
from pyspark.sql.functions import col, floor, row_number, min
from pyspark.sql.window import Window

# 1. กำหนดความละเอียดของพิกัด (Lat/Lon)
# การคูณด้วย 10,000 และใช้ floor จะทำให้ Lat/Lon มีความละเอียดประมาณ 10-20 เมตร
df_grouped = df_clean.withColumn("micro_lat", floor(col("lat") * 10000)) \
                        .withColumn("micro_lon", floor(col("lon") * 10000)) \
                        .withColumn("comment_group", col("comment_clean")) # ใช้ comment_clean เป็นกลุ่มหลัก

# 2. จัดอันดับ Ticket ภายในกลุ่มที่ซ้ำกัน
# W: จัดกลุ่มตามพิกัดและข้อความที่เหมือนกัน
window_spec = Window.partitionBy("micro_lat", "micro_lon", "comment_group").orderBy(col("timestamp_dt").asc())

df_ranked = df_grouped.withColumn(
    "rank", 
    row_number().over(window_spec)
)

# 3. กรอง: เก็บเฉพาะ Ticket แรกที่ถูกรายงาน (rank = 1)
df_deduplicated_final = df_ranked.filter(col("rank") == 1).drop("micro_lat", "micro_lon", "comment_group", "rank")

print(f"จำนวน Ticket ก่อน Deduplication: {df_clean.count()}")
print(f"จำนวน Ticket หลัง Deduplication: {df_deduplicated_final.count()}")

df_deduplicated_final.filter(col("district") == "พระโขนง").show(5)

จำนวน Ticket ก่อน Deduplication: 1667
จำนวน Ticket หลัง Deduplication: 1644
+-----------+--------------------+--------------------+--------------------+--------------------+-----------+------------------+--------------------+-----------+--------+-------------+--------------------+--------------+----+------------+--------------------+---------+--------+---------+--------+-------------------+-------------------+-------------------+-------------------+--------------+------------------+---------+--------------------+
|  ticket_id|                type|        organization|             comment|               photo|photo_after|            coords|             address|subdistrict|district|     province|           timestamp|         state|star|count_reopen|       last_activity|  lon_raw| lat_raw|      lon|     lat|      timestamp_str|  last_activity_str|       timestamp_dt|   last_activity_dt|timestamp_date|last_activity_date|DaysToFix|       comment_clean|
+-----------+--------------------+----

In [114]:
from pyspark.sql.functions import col, desc

# สมมติว่า df_final_ml คือ DataFrame ที่คุณใช้ล่าสุด

# 1. จัดกลุ่มตามคอลัมน์ comment_clean และนับจำนวนแถวในแต่ละกลุ่ม
df_comment_counts = df_deduplicated_final.groupBy("comment_clean").count()

# 2. เรียงลำดับจากจำนวนนับ (count) ที่มากที่สุดไปน้อยที่สุด (Descending)
df_duplicate_summary = df_comment_counts.orderBy(
    col("count").desc()
)

# 3. แสดงผลลัพธ์ 20 อันดับแรก ที่มีจำนวนซ้ำกันมากกว่า 1 ครั้ง
print("--- 20 อันดับแรกของข้อความร้องเรียนที่ซ้ำกันมากที่สุด ---")
df_duplicate_summary.filter(col("count") > 1).show(
    20, 
    truncate=False # แสดงข้อความ comment_clean แบบเต็ม
)

--- 20 อันดับแรกของข้อความร้องเรียนที่ซ้ำกันมากที่สุด ---
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|comment_clean                                                                                                                                                                                                                                                                 |count|
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|แจ้ง จักรยานยนต์ขับขี่บนทางเท้า ระหว่างซอยเพชรเกษม 62/1 62/2 62/3                                       

In [115]:

# @udf(ArrayType(StringType()))
# def thai_tokenize(text):
#     if text and text.strip():
#         # ใช้ Engine 'newmm' และลบช่องว่างออกจาก tokens
#         return word_tokenize(text, engine='newmm', keep_whitespace=False)
#     else:
#         return []

# # UDF สำหรับการลบ Stopwords
# # โหลด Stopwords List แค่ครั้งเดียว
# STOPWORDS = set(thai_stopwords())
# @udf(ArrayType(StringType()))
# def remove_stopwords(tokens):
#     if tokens is not None:
#         return [word for word in tokens if word not in STOPWORDS and word != '']
#     else:
#         return []


# # A. สร้างคอลัมน์ tokens จาก 'comment_clean'
# df_tokenized = df_clean.withColumn(
#     "tokens", 
#     thai_tokenize(col("comment_clean")) # แก้ไขเป็น 'comment_clean' แล้ว
# )

# # B. สร้างคอลัมน์ final_tokens (สำหรับ LLM Sentiment)
# df_final_llm = df_tokenized.withColumn(
#     "final_tokens", 
#     remove_stopwords(col("tokens"))
# )

# df_final_llm.select("comment_clean", "final_tokens").show(5, truncate=False)

In [116]:
# df_clean.describe().show()

In [117]:
# คอลัมน์ที่เราต้องการเก็บไว้เท่านั้น
COLUMNS_TO_KEEP = [
    "ticket_id",
    "type",
    # "organization",
    # "state",
    "address",
    "district",
    
    # Core features
    "lat",
    "lon",
    "DaysToFix",
    
    # Text Input for LLM
    "comment_clean",
    
    # Time Analysis (DateTimes)
    "timestamp_dt",
    "last_activity_dt"
]

df_ready_for_export = df_deduplicated_final.select(*COLUMNS_TO_KEEP)
df_ready_for_export = df_ready_for_export.na.drop(subset=["comment_clean", "district"])
df_ready_for_export = df_ready_for_export.filter(
    col("district").isNotNull() 
)
print(f"จำนวนคอลัมน์เดิม: {len(df_deduplicated_final.columns)}")
print(f"จำนวนคอลัมน์ใหม่: {len(df_ready_for_export.columns)}")

df_ready_for_export.printSchema()

จำนวนคอลัมน์เดิม: 28
จำนวนคอลัมน์ใหม่: 10
root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- DaysToFix: integer (nullable = true)
 |-- comment_clean: string (nullable = true)
 |-- timestamp_dt: timestamp (nullable = true)
 |-- last_activity_dt: timestamp (nullable = true)



In [118]:

df_ready_for_export.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_ready_for_export.columns
]).show()


+---------+----+-------+--------+---+---+---------+-------------+------------+----------------+
|ticket_id|type|address|district|lat|lon|DaysToFix|comment_clean|timestamp_dt|last_activity_dt|
+---------+----+-------+--------+---+---+---------+-------------+------------+----------------+
|        0|   0|      0|       0|  0|  0|        0|            0|           0|               0|
+---------+----+-------+--------+---+---+---------+-------------+------------+----------------+



In [119]:
from pyspark.sql.functions import lit



# ตอนนี้ DaysToFix และ is_fixed สามารถใช้เป็น Features ในโมเดลได้
# df_model_ready.select("DaysToFix", "is_fixed","state", "comment_clean").show(10)

df_multi_hot = df_ready_for_export

for t in livability_types:
    # สร้างชื่อคอลัมน์ใหม่ (เช่น type_ถนน)
    new_col_name = f"type_{t}"
    
    df_multi_hot = df_multi_hot.withColumn(
        new_col_name,
        # ถ้าคอลัมน์ 'type' ดั้งเดิม มีข้อความ 't' อยู่ ให้กำหนดค่าเป็น 1
        when(col("type").contains(t), lit(1)).otherwise(lit(0))
    )

# --- ตรวจสอบผลลัพธ์ ---
# เลือกคอลัมน์ type ดั้งเดิม และคอลัมน์ Multi-Hot ที่สร้างขึ้นใหม่
selected_cols = ["ticket_id", "type"] + [f"type_{t}" for t in livability_types]
selected_cols.pop(-3)
selected_cols.append("type_PM25")
df_multi_hot = df_multi_hot.withColumnRenamed("type_PM2", "type_PM25")

df_multi_hot.select(*selected_cols).show(20, truncate=False)

+-----------+---------------------------------------------------------------------------+--------+------------+----------------+-------------+--------------+------------+----------------+------------+-----------+----------+----------+---------+
|ticket_id  |type                                                                       |type_ถนน|type_ทางเท้า|type_ความปลอดภัย|type_แสงสว่าง|type_ความสะอาด|type_กีดขวาง|type_ท่อระบายน้ำ|type_น้ำท่วม|type_ต้นไม้|type_จราจร|type_สะพาน|type_PM25|
+-----------+---------------------------------------------------------------------------+--------+------------+----------------+-------------+--------------+------------+----------------+------------+-----------+----------+----------+---------+
|2025-J2PUCL|ฝุ่นควัน&กลิ่น&PM2.5                                                       |0       |0           |0               |0            |0             |0           |0               |0           |0          |0         |0         |1        |
|2025-4PA46X|ผิดกฎจร

In [120]:
from pyspark.sql.functions import col, year

# สร้างคอลัมน์ 'year_reported' จาก timestamp_dt
df_final_year = df_multi_hot.withColumn(
    "year_reported", 
    year(col("timestamp_dt"))
)

# สร้างคอลัมน์ 'year_last_activity' จาก last_activity_dt
df_final_year = df_final_year.withColumn(
    "year_last_activity", 
    year(col("last_activity_dt"))
)

In [121]:
df_final_ml = df_final_year.drop("type") 
# ถ้าคุณสร้างคอลัมน์ 'type_clean' ชั่วคราวในการแก้ไขปัญหา ก็ควรลบคอลัมน์นั้นด้วย
# df_final_ml = df_multi_hot.drop("type", "state", "type_clean") 
df_final_ml = df_final_ml.withColumnRenamed("DaysToFix", "DaysActive_Pending")
# print("ตัวอย่าง Schema หลัง Drop:")
df_final_ml.printSchema()

print(df_final_ml.count())


root
 |-- ticket_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- district: string (nullable = true)
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- DaysActive_Pending: integer (nullable = true)
 |-- comment_clean: string (nullable = true)
 |-- timestamp_dt: timestamp (nullable = true)
 |-- last_activity_dt: timestamp (nullable = true)
 |-- type_ถนน: integer (nullable = false)
 |-- type_ทางเท้า: integer (nullable = false)
 |-- type_ความปลอดภัย: integer (nullable = false)
 |-- type_แสงสว่าง: integer (nullable = false)
 |-- type_ความสะอาด: integer (nullable = false)
 |-- type_กีดขวาง: integer (nullable = false)
 |-- type_ท่อระบายน้ำ: integer (nullable = false)
 |-- type_น้ำท่วม: integer (nullable = false)
 |-- type_ต้นไม้: integer (nullable = false)
 |-- type_PM25: integer (nullable = false)
 |-- type_จราจร: integer (nullable = false)
 |-- type_สะพาน: integer (nullable = false)
 |-- year_reported: integer (nullable = true)
 |-- year_last_a

In [124]:


from pyspark.sql.functions import col, regexp_replace

# 1. กำหนดคอลัมน์ที่เป็น String และมีโอกาสเกิด Newline
# ให้ใส่ชื่อคอลัมน์ที่เป็นข้อความขนาดใหญ่ทั้งหมดที่คุณมี (เช่น comment_clean, address)
string_cols_to_clean = ["comment_clean", "address", "district"] 

# 2. ทำการวนซ้ำเพื่อแทนที่อักขระ Newline ในทุกคอลัมน์
df_cleaned_for_export = df_final_ml 

for col_name in string_cols_to_clean:
    # แทนที่อักขระขึ้นบรรทัดใหม่ (\n) และ Carriage Return (\r) ด้วยช่องว่าง ' '
    df_cleaned_for_export = df_cleaned_for_export.withColumn(
        col_name,
        regexp_replace(col(col_name), "[\r\n]", " ")
    )

print("--- ล้างอักขระขึ้นบรรทัดใหม่เสร็จสิ้น ---")
# 3. ใช้ DataFrame นี้ในการบันทึกไฟล์
OUTPUT_PATH = r"C:\Users\sasit\CU\2-1\dsde\project\dsdengdeng-project-dsde\data\processed\traffy-fondue\traffy_fondue_bangkok_processed.csv"
import pandas as pd


# แปลง Spark DataFrame → pandas DataFrame
pdf = df_cleaned_for_export.toPandas()

# เซฟเป็น CSV ด้วย pandas (ไม่ผ่าน Hadoop)
pdf.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("✅ Saved CSV to:", OUTPUT_PATH)


--- ล้างอักขระขึ้นบรรทัดใหม่เสร็จสิ้น ---
✅ Saved CSV to: C:\Users\sasit\CU\2-1\dsde\project\dsdengdeng-project-dsde\data\processed\traffy-fondue\traffy_fondue_bangkok_processed.csv


In [ ]:
df_map_sample_pandas = df_final_ml.sample(
    fraction=10000 / df_ready_for_export.count(), 
    seed=42
).toPandas()
display(df_map_sample_pandas.head(10))

,ticket_id,address,district,lat,lon,DaysActive_Pending,comment_clean,timestamp_dt,last_activity_dt,type_ถนน,...,type_ความสะอาด,type_กีดขวาง,type_ท่อระบายน้ำ,type_น้ำท่วม,type_ต้นไม้,type_PM25,type_จราจร,type_สะพาน,year_reported,year_last_activity
0,FTHAWF,68/43 ซอย แสมดำ 14 แขวงแสมดำ เขตบางขุนเทียน กร...,บางขุนเทียน,13.59925,100.383110,889,ปัญหา แจ้งปัญหาน้ำท่วมถนน เวลาฝนตกแล้วมีน้ำท่ว...,2023-07-01 11:15:24,2024-08-02 02:33:36,1,...,0,0,1,1,0,0,0,0,2023,2024
1,2024-F8UB8P,508/2 ซ. อนามัยงามเจริญ แขวงท่าข้าม เขตบางขุนเ...,บางขุนเทียน,13.60742,100.459160,563,ร้องเรียนเรื่องถนนเส้นอนามัยงามเจริญ ฝั่งหลังว...,2024-05-22 02:54:55,2024-12-21 06:41:44,1,...,0,0,0,0,0,0,0,0,2024,2024
2,2024-MMATB3,JF55+5WM ซ. อนามัยงามเจริญ แขวงท่าข้าม เขตบางข...,บางขุนเทียน,13.60794,100.459892,425,น้ำท่วมมากมาย ความสูงระดับหน้าแข้ง น้ำท่วมมาแล...,2024-10-07 06:23:52,2024-10-24 07:05:07,0,...,0,0,0,1,0,0,0,0,2024,2024
3,2024-7XX6BP,ชุมชนโฟร์โมสต์ แขวงแสมดำ เขตบางขุนเทียน กรุงเท...,บางขุนเทียน,13.62597,100.393013,473,เส้นถนนพระราม 2 ที่ทำทางด่วนอยู่ มีสิ่งก่อสร้า...,2024-08-20 02:16:13,2024-08-28 04:39:50,1,...,0,0,0,0,0,0,0,0,2024,2024
4,PP8DD2,หมู่บ้านบุราสิริ ท่าข้าม-พระราม 2 แขวงท่าข้าม ...,บางขุนเทียน,13.62664,100.445992,553,ปัญหา บนถนนดังกล่าว หน้าหมู่บ้านบุราสิริ ท่าข้...,2024-06-01 06:51:29,2024-06-19 06:22:13,1,...,0,1,0,0,0,0,1,1,2024,2024
5,2022-CT627P,JCGV+QGH แขวง ท่าข้าม เขตบางขุนเทียน กรุงเทพมห...,บางขุนเทียน,13.62683,100.443840,1204,เขตบางขุนเทียน ถนนเลียบทางด่วนกาญจนา,2022-08-20 10:41:55,2022-08-20 11:17:15,1,...,0,0,0,0,0,0,0,0,2022,2022
6,RUL4V2,"146 Moo4 Soi Prachauthit72Prachauthit Rd.,, Kw...",ทุ่งครุ,13.62909,100.492340,624,ปัญหา ในซอยดังกล่าว หน้าร้านก๋วยเตี๋ยว ไม่ทราบ...,2024-03-22 07:04:06,2024-12-17 03:35:06,1,...,0,0,0,0,0,0,0,0,2024,2024
7,2024-NRQPEM,3 ถนน พรมแดน บางบอนใต้ เขตบางบอน กรุงเทพมหานคร...,บางบอน,13.63214,100.374763,546,ขอให้ตรวจสอบอู่ซ่อมรถยนต์ชื่อนี้มีใบอนุญาตหรือ...,2024-06-08 00:37:05,2024-12-31 17:01:15,0,...,0,1,0,0,0,0,0,0,2024,2024
8,2023-H7DTQP,JCM8+M7J แขวงแสมดำ เขตบางขุนเทียน กรุงเทพมหานค...,บางขุนเทียน,13.63437,100.415611,841,น้ำรั่วจากสะพานไหลลงมาทำให้พื้นหน้าบ้านเฉอะแฉะ,2023-08-18 03:40:33,2023-08-18 07:59:41,0,...,0,0,0,0,0,0,0,1,2023,2023
9,AD63V9,835 ซอย หมู่บ้านวิเศษสุขนคร 18/16 แขวงทุ่งครุ ...,ทุ่งครุ,13.63702,100.512077,330,ปัญหา ภายในหมู่บ้านดังกล่าว พบดวงไฟฟ้าส่องสว่า...,2025-01-10 03:53:58,2025-01-10 04:06:39,1,...,0,0,0,0,0,0,0,0,2025,2025


In [ ]:
# # -----------------------------
# # 1️⃣ Import libraries
# # -----------------------------
# from pyspark.sql.functions import col, split
# import pandas as pd
# import numpy as np
# import folium
# from folium.plugins import HeatMap

# # -----------------------------
# # 2️⃣ แยก lon / lat จาก coords string
# # -----------------------------
# # สมมติ coords เป็น "lon,lat"
# df_final_ready = df_final_ready.withColumn("lon", split(col("coords"), ",").getItem(0).cast("double")) \
#                                .withColumn("lat", split(col("coords"), ",").getItem(1).cast("double"))

# # -----------------------------
# # 3️⃣ Sample data (10,000 rows) และแปลงเป็น Pandas
# # -----------------------------
# n_sample = 10000
# n_total = df_final_ready.count()
# df_sample_pandas = df_final_ready.sample(fraction=n_sample / n_total, seed=42).toPandas()

# # -----------------------------
# # 4️⃣ เตรียม weight (DaysToFix) แบบ log scale
# # -----------------------------
# df_sample_pandas['weight'] = np.log1p(df_sample_pandas['DaysToFix'])

# # -----------------------------
# # 5️⃣ เตรียมข้อมูลสำหรับ HeatMap
# # -----------------------------
# data_heatmap = df_sample_pandas[['lat', 'lon', 'weight']].values.tolist()

# # -----------------------------
# # 6️⃣ สร้าง Folium Map
# # -----------------------------
# center_lat = 13.737
# center_lon = 100.528

# m = folium.Map(
#     location=[center_lat, center_lon],
#     zoom_start=11,
#     tiles="cartodbpositron"
# )

# # -----------------------------
# # 7️⃣ เพิ่ม HeatMap Layer
# # -----------------------------
# HeatMap(
#     data_heatmap,
#     radius=10,            # ขนาดจุด
#     blur=15,              # ความฟุ้ง
#     max_val=df_sample_pandas['weight'].max()
# ).add_to(m)

# # -----------------------------
# # 8️⃣ แสดงผล (Jupyter Notebook) / บันทึกเป็น HTML
# # -----------------------------
# m  # Interactive map in notebook
# m.save("daystofix_heatmap.html")  # บันทึกเป็น HTML
